# Ongoing Assessment 1
## Part I - Connect to a database using Python

**Approved implementation: SQLite on the local computer.** The student confirmed
that the lecturer allows SQLite as an alternative to IBM Db2 Cloud.
All saved query results in this notebook were produced locally using SQLite.
The optional Db2 section contains live connection code, but has not been run on IBM Cloud.

This notebook demonstrates library imports, a persistent database connection,
cursor-based SQL queries, data retrieval and display, CSV export, and connection closure.
It uses the original CSV supplied with the assessment. No other student's code or
saved results are used.

Run this notebook from its repository folder using **Run All**.

### 1. Import the libraries

`sqlite3` is Python's built-in SQLite library. `pandas` formats the returned records
for display. The accompanying `database_lab.py` contains the reusable connection,
cursor, query, export and verification functions.

In [1]:
from pathlib import Path
import json
import sqlite3
import pandas as pd
from IPython.display import display, Markdown
from database_lab import ROOT, RAW_CSV, COLUMNS, source_records, query_specs, run_sqlite_lab

print("SQLite library imported. SQLite version:", sqlite3.sqlite_version)
print("Dataset:", RAW_CSV.name)

SQLite library imported. SQLite version: 3.53.1
Dataset: cardio_train_raw.csv


### 2. Inspect the supplied dataset

The file uses semicolons as separators. Age is in **days**, height in centimetres,
and weight in kilograms. `CARDIO` is the binary target. `ID` identifies the record.
Preserve raw values for the database lab; converting age for display must not alter
the source. Numeric gender codes are retained without assigning unverified labels.

In [2]:
records = source_records()
raw = pd.DataFrame(records, columns=COLUMNS)
print(f"Source contains {len(raw):,} records and {len(raw.columns)} columns.")
display(raw.head(10))
display(raw.isna().sum().rename("MISSING_VALUES").to_frame())
print("Unique IDs:", raw["ID"].nunique())
print("Disease cases:", int(raw["CARDIO"].sum()))
print("Age remains in days in the database and CSV export.")

Source contains 70,000 records and 13 columns.


,ID,AGE,GENDER,HEIGHT,WEIGHT,AP_HI,AP_LO,CHOLESTEROL,GLUC,SMOKE,ALCO,ACTIVE,CARDIO
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
5,8,21914,1,151,67.0,120,80,2,2,0,0,0,0
6,9,22113,1,157,93.0,130,80,3,1,0,0,1,0
7,12,22584,2,178,95.0,130,90,3,3,0,0,1,1
8,13,17668,1,158,71.0,110,70,1,1,0,0,1,0
9,14,19834,1,164,68.0,110,60,1,1,0,0,0,0


,MISSING_VALUES
ID,0
AGE,0
GENDER,0
HEIGHT,0
WEIGHT,0
AP_HI,0
AP_LO,0
CHOLESTEROL,0
GLUC,0
SMOKE,0


Unique IDs: 70000
Disease cases: 34979
Age remains in days in the database and CSV export.


### 3. Connect, create the table and retrieve data using a cursor

The database is stored at `data/cardio_train.sqlite`. On the first run, the supplied
records are inserted into `CARDIO_TRAIN`. On later runs, the existing records are
compared with the source CSV and reused. `ID` is the primary key.

The workflow uses these Python operations:

```python
connection = sqlite3.connect(database_path)
cursor = connection.cursor()
try:
    cursor.execute("SELECT COUNT(*) AS TOTAL_ROWS FROM CARDIO_TRAIN")
    rows = cursor.fetchall()
finally:
    cursor.close()
    connection.close()
```

The actual function below performs all queries, saves each result, exports the complete
table and closes the connection in a `finally` block. It then reopens the database
read-only and verifies integrity and persistence.

In [3]:
results, status = run_sqlite_lab()
print("Connected to the local SQLite database and completed the SQL workflow.")
print("Database:", "data/cardio_train.sqlite")
print("Connection closed:", status["connection_closed"])
print("Reopened database integrity:", status["integrity_check"])
print("Every source record matches the saved database:", status["all_source_records_match"])

Connected to the local SQLite database and completed the SQL workflow.
Database: data/cardio_train.sqlite
Connection closed: True
Reopened database integrity: ok
Every source record matches the saved database: True


### 4. SQL queries and retrieved results

Q01-Q10 follow the ten Python API query exercises in the brief. Q11-Q13 add the
preview, lowest diastolic blood pressure and alcohol queries.
Every selection of ten records uses a deterministic order with ID to break ties.
The same queries are also saved under `sql/queries_sqlite.sql`.

For selected samples, the disease percentage is
`100 * records_with_CARDIO_1 / number_of_returned_records`.
This describes the selected sample; it does not estimate an individual's probability
of cardiovascular disease. Glucose codes indicate measured glucose categories,
not glucose consumption, and the smoking indicator does not measure frequency.

#### Q01. Total patient records

```sql
SELECT COUNT(*) AS TOTAL_ROWS FROM CARDIO_TRAIN;
```

In [4]:
result = next(item for item in results if item["id"] == "Q01")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,TOTAL_ROWS
0,70000


#### Q02. Records with cardiovascular disease

```sql
SELECT COUNT(*) AS DISEASE_CASES FROM CARDIO_TRAIN WHERE CARDIO = 1;
```

In [5]:
result = next(item for item in results if item["id"] == "Q02")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,DISEASE_CASES
0,34979


#### Q03. Patient counts by gender code

```sql
SELECT GENDER, COUNT(*) AS PATIENTS FROM CARDIO_TRAIN GROUP BY GENDER ORDER BY GENDER;
```

In [6]:
result = next(item for item in results if item["id"] == "Q03")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,GENDER,PATIENTS
0,1,45530
1,2,24470


#### Q04. Ten highest systolic blood pressure values

```sql
SELECT ID, AP_HI, CARDIO FROM CARDIO_TRAIN ORDER BY AP_HI DESC, ID LIMIT 10;
```

In [7]:
result = next(item for item in results if item["id"] == "Q04")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,AP_HI,CARDIO
0,58374,16020,1
1,36339,14020,1
2,36414,14020,1
3,66998,14020,1
4,67502,14020,0
5,79116,13010,1
6,79679,13010,0
7,11089,11500,1
8,73356,11020,1
9,99089,2000,1


Cases with disease: 8/10
Observed disease percentage: 80.0%


#### Q05. Ten heaviest patients above the average weight

```sql
SELECT ID, WEIGHT, CARDIO FROM CARDIO_TRAIN WHERE WEIGHT > (SELECT AVG(WEIGHT) FROM CARDIO_TRAIN) ORDER BY WEIGHT DESC, ID LIMIT 10;
```

In [8]:
result = next(item for item in results if item["id"] == "Q05")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,WEIGHT,CARDIO
0,618,200.0,0
1,71945,200.0,1
2,87498,183.0,1
3,52564,181.0,1
4,6716,180.0,1
5,64829,180.0,1
6,86506,180.0,1
7,87612,180.0,1
8,28683,178.0,0
9,39156,178.0,1


Cases with disease: 8/10
Observed disease percentage: 80.0%


#### Q06. Ten patients with glucose above normal

```sql
SELECT ID, GLUC, CARDIO FROM CARDIO_TRAIN WHERE GLUC > 1 ORDER BY ID LIMIT 10;
```

In [9]:
result = next(item for item in results if item["id"] == "Q06")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,GLUC,CARDIO
0,8,2,0
1,12,3,1
2,27,3,0
3,32,2,1
4,40,3,0
5,68,3,1
6,71,2,0
7,92,2,1
8,103,2,0
9,117,2,1


Cases with disease: 5/10
Observed disease percentage: 50.0%


#### Q07. Ten oldest patients (age in days)

```sql
SELECT ID, AGE, AGE / 365.0 AS AGE_YEARS, CARDIO FROM CARDIO_TRAIN ORDER BY AGE DESC, ID LIMIT 10;
```

In [10]:
result = next(item for item in results if item["id"] == "Q07")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,AGE,AGE_YEARS,CARDIO
0,81643,23713,64.967123,1
1,97124,23701,64.934247,1
2,72373,23692,64.909589,1
3,29892,23690,64.904110,1
4,52297,23687,64.895890,1
5,53805,23684,64.887671,1
6,57321,23678,64.871233,0
7,98749,23677,64.868493,1
8,25799,23675,64.863014,1
9,82562,23675,64.863014,1


Cases with disease: 9/10
Observed disease percentage: 90.0%


#### Q08. Ten patients with cholesterol code 3

```sql
SELECT ID, CHOLESTEROL, CARDIO FROM CARDIO_TRAIN WHERE CHOLESTEROL = 3 ORDER BY ID LIMIT 10;
```

In [11]:
result = next(item for item in results if item["id"] == "Q08")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,CHOLESTEROL,CARDIO
0,1,3,1
1,2,3,1
2,9,3,0
3,12,3,1
4,35,3,1
5,42,3,1
6,43,3,1
7,46,3,1
8,59,3,1
9,68,3,1


Cases with disease: 9/10
Observed disease percentage: 90.0%


#### Q09. Ten patients who smoke

```sql
SELECT ID, SMOKE, CARDIO FROM CARDIO_TRAIN WHERE SMOKE = 1 ORDER BY ID LIMIT 10;
```

In [12]:
result = next(item for item in results if item["id"] == "Q09")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,SMOKE,CARDIO
0,23,1,0
1,29,1,0
2,52,1,1
3,73,1,1
4,74,1,1
5,90,1,1
6,138,1,0
7,140,1,1
8,142,1,1
9,166,1,1


Cases with disease: 7/10
Observed disease percentage: 70.0%


#### Q10. Ten physically active patients

```sql
SELECT ID, ACTIVE, CARDIO FROM CARDIO_TRAIN WHERE ACTIVE = 1 ORDER BY ID LIMIT 10;
```

In [13]:
result = next(item for item in results if item["id"] == "Q10")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,ACTIVE,CARDIO
0,0,1,0
1,1,1,1
2,3,1,1
3,9,1,0
4,12,1,1
5,13,1,0
6,15,1,0
7,16,1,0
8,21,1,0
9,23,1,0


Cases with disease: 3/10
Observed disease percentage: 30.0%


#### Q11. Preview ten patient records

```sql
SELECT * FROM CARDIO_TRAIN ORDER BY ID LIMIT 10;
```

In [14]:
result = next(item for item in results if item["id"] == "Q11")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,AGE,GENDER,HEIGHT,WEIGHT,AP_HI,AP_LO,CHOLESTEROL,GLUC,SMOKE,ALCO,ACTIVE,CARDIO
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
5,8,21914,1,151,67.0,120,80,2,2,0,0,0,0
6,9,22113,1,157,93.0,130,80,3,1,0,0,1,0
7,12,22584,2,178,95.0,130,90,3,3,0,0,1,1
8,13,17668,1,158,71.0,110,70,1,1,0,0,1,0
9,14,19834,1,164,68.0,110,60,1,1,0,0,0,0


Cases with disease: 4/10
Observed disease percentage: 40.0%


#### Q12. Ten lowest diastolic blood pressure values

```sql
SELECT ID, AP_LO, CARDIO FROM CARDIO_TRAIN ORDER BY AP_LO, ID LIMIT 10;
```

In [15]:
result = next(item for item in results if item["id"] == "Q12")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,AP_LO,CARDIO
0,85816,-70,1
1,2845,0,0
2,19258,0,0
3,23512,0,1
4,24837,0,0
5,32749,0,1
6,34120,0,1
7,36325,0,0
8,39577,0,1
9,45400,0,0


Cases with disease: 5/10
Observed disease percentage: 50.0%


#### Q13. Ten patients who drink alcohol

```sql
SELECT ID, ALCO, CARDIO FROM CARDIO_TRAIN WHERE ALCO = 1 ORDER BY ID LIMIT 10;
```

In [16]:
result = next(item for item in results if item["id"] == "Q13")
display(pd.DataFrame(result["rows"], columns=result["columns"]))
if "disease_cases" in result:
    print(f'Cases with disease: {result["disease_cases"]}/{result["sample_size"]}')
    print(f'Observed disease percentage: {result["disease_percent"]:.1f}%')

,ID,ALCO,CARDIO
0,23,1,0
1,81,1,0
2,134,1,1
3,136,1,0
4,142,1,1
5,180,1,1
6,199,1,0
7,213,1,1
8,240,1,0
9,318,1,1


Cases with disease: 5/10
Observed disease percentage: 50.0%


### 5. Export for offline analysis and verify the closed connection

The export is written from the SQL result set rather than copied from the input CSV.
It has comma separators, the original 13 columns, age in days and no extra index
column. Every exported value is checked against the source. The saved SQLite file
can be opened again after the original connection has been closed.

In [17]:
export = pd.read_csv(ROOT / "data" / "CARDIO_TRAIN_export.csv")
print("SQL export:", "data/CARDIO_TRAIN_export.csv")
print("Exported records:", len(export))
print("Export matches every source record:", status["export_matches_source"])
print("Original connection closed:", status["connection_closed"])
print("Verification connection closed:", status["reopened_connection_closed"])
display(pd.DataFrame([status]))

SQL export: data/CARDIO_TRAIN_export.csv
Exported records: 70000
Export matches every source record: True
Original connection closed: True
Verification connection closed: True


,engine,cloud_executed,lecturer_approved_local_alternative,source_sha256,executed_at_utc,all_source_records_match,row_count,disease_cases,integrity_check,connection_closed,reopened_connection_closed,export_matches_source,executed_queries,export_rows,success
0,SQLite,False,True,21a705d23381b0dfd6a6416da701b490744f1fc3b47e9f...,2026-10-07T21:27:44.789157+00:00,True,70000,34979,ok,True,True,True,13,70000,True


### 6. Findings from the local database run

The original dataset contains 70,000 records. Of these, 34,979 have `CARDIO = 1`
and 35,021 have `CARDIO = 0`; the observed disease proportion is about 49.97%.
The counts by gender code are 45,530 for code 1 and 24,470 for code 2.

The highest systolic pressure query includes implausibly large raw values, including
16,020. These values are retained to reproduce the raw-data exercise and should be
investigated during the later data-cleaning stage. Query Q07 reports ages in both
days and approximate years, without changing the stored data.

The query outputs, export, closure checks and database integrity result provide local
execution evidence. This submission does not claim that a Db2 Cloud connection or
a Watson Studio session was executed.

### 7. Optional IBM Db2 Cloud connection

This section preserves the original Cloud connection objective as an optional path.
It is skipped by default and is not needed for the approved SQLite version.

The helper imports `ibm_db` and `ibm_db_dbi`, builds an SSL DSN, calls
`ibm_db.connect(dsn, "", "")`, retrieves server and client metadata using
`ibm_db.server_info` and `ibm_db.client_info`, retrieves the table through a DB-API
cursor, exports it, and calls `ibm_db.close` in a `finally` block.

For actual Cloud execution, first import the raw CSV into `CARDIO_TRAIN` in your
Db2 instance and install `requirements-db2.txt`. Set the flags below to enable
the connection and the private credential prompts. Environment variables can
be used instead. No credentials from the assignment PDF are used.

```python
dsn = (
    "DRIVER={IBM DB2 ODBC DRIVER};"
    "DATABASE=...;HOSTNAME=...;PORT=...;PROTOCOL=TCPIP;"
    "UID=...;PWD=...;SECURITY=SSL;"
)
conn = ibm_db.connect(dsn, "", "")
try:
    server = ibm_db.server_info(conn)
    client = ibm_db.client_info(conn)
finally:
    ibm_db.close(conn)
```

Actual implementation: `database_lab.py`, function `run_db2_lab`.
The connection string and password are never printed. Cloud exports and evidence
are stored separately from the local results.

In [18]:
RUN_DB2 = False
ENTER_DB2_CREDENTIALS = False

if RUN_DB2:
    import os
    from getpass import getpass
    import ibm_db
    from database_lab import run_db2_lab

    if ENTER_DB2_CREDENTIALS:
        os.environ["DB2_DATABASE"] = input("Database name (e.g. BLUDB): ").strip()
        os.environ["DB2_HOSTNAME"] = input("IBM Db2 hostname: ").strip()
        os.environ["DB2_PORT"] = input("IBM SSL port: ").strip()
        os.environ["DB2_UID"] = input("Database username: ").strip()
        os.environ["DB2_PWD"] = getpass("Database password: ")
        os.environ["DB2_TABLE"] = input("Table (Enter for CARDIO_TRAIN): ").strip() or "CARDIO_TRAIN"
        certificate = input("Certificate file if required (otherwise Enter): ").strip()
        os.environ["DB2_SSL_CERT"] = certificate

    cloud_results, cloud_status = run_db2_lab()
    for item in cloud_results:
        display(Markdown(f'**{item["id"]}: {item["title"]}**'))
        display(pd.DataFrame(item["rows"], columns=item["columns"]))
    print("Cloud queries and export completed:", cloud_status["success"])
else:
    print("Db2 Cloud execution skipped. Saved assessment results use approved local SQLite.")

Db2 Cloud execution skipped. Saved assessment results use approved local SQLite.


### Completion checklist

| Objective | Evidence |
| --- | --- |
| Import Python database library | Section 1, `sqlite3` |
| Create a connection | Section 3, persistent SQLite database |
| Query using a cursor and display data | Section 4, executed Q01-Q13 |
| Retrieve data for offline analysis | Section 5, verified SQL CSV export |
| Close the database connection | Section 5, closure and reopen checks |
| Original Db2 library and credentials path | Section 7, optional code, not Cloud execution evidence |

**The approved local database workflow is complete.** Later machine learning and
analysis exercises are outside this Part I notebook.

References: the supplied ADY201m brief and CSV;
[Python sqlite3](https://docs.python.org/3/library/sqlite3.html);
[IBM Python Db2 driver](https://github.com/ibmdb/python-ibmdb);
[IBM API documentation](https://github.com/ibmdb/python-ibmdb/wiki/APIs).